In [ ]:
!pip install transformers torch scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import re, warnings
warnings.filterwarnings('ignore')

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", DEVICE)

Device: cpu


In [ ]:
train = pd.read_csv('Train.csv')
test  = pd.read_csv('Test.csv')

# Drop the 1 row with missing label
train = train.dropna(subset=['label']).reset_index(drop=True)

print("Train:", train.shape, "| Test:", test.shape)
print(train['label'].value_counts())

def clean_text(text):
    text = str(text)
    text = re.sub(r'<url>', '', text)            # already anonymized URLs
    text = re.sub(r'<user>', '@user', text)      # already anonymized users
    text = re.sub(r'&amp;', '&', text)           # fix HTML entities
    text = re.sub(r'[^\w\s@#!?.,&]', '', text)  # remove noise
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train['clean_text'] = train['safe_text'].apply(clean_text)
test['clean_text']  = test['safe_text'].apply(clean_text)

Train: (10000, 4) | Test: (5177, 2)
label
 0.000000    4908
 1.000000    4053
-1.000000    1038
 0.666667       1
Name: count, dtype: int64


In [ ]:
# Best model for tweets — pre-trained on 124M tweets
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment-latest"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

class TweetDataset(Dataset):
    def __init__(self, texts, targets=None, max_len=128):
        self.texts   = texts
        self.targets = targets
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        item = {
            'input_ids':      enc['input_ids'].squeeze(),
            'attention_mask': enc['attention_mask'].squeeze(),
        }
        if self.targets is not None:
            item['target'] = torch.tensor(self.targets[idx], dtype=torch.float)
        return item

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [ ]:
class SentimentRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(MODEL_NAME)
        self.drop = nn.Dropout(0.3)
        self.fc   = nn.Linear(self.backbone.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        out  = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        pool = out.last_hidden_state[:, 0, :]  # CLS token
        pool = self.drop(pool)
        return self.fc(pool).squeeze(-1)

In [ ]:
EPOCHS  = 4
BATCH   = 16
LR      = 2e-5
N_FOLDS = 5

texts  = train['clean_text'].tolist()
labels = train['label'].tolist()

oof_preds  = np.zeros(len(train))
test_preds = np.zeros(len(test))

# Stratify on label (−1, 0, 1) to preserve class balance in each fold
strat = train['label'].round().astype(int)

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

for fold, (tr_idx, val_idx) in enumerate(skf.split(texts, strat)):
    print(f"\n========== FOLD {fold+1}/{N_FOLDS} ==========")

    tr_texts   = [texts[i]  for i in tr_idx]
    val_texts  = [texts[i]  for i in val_idx]
    tr_labels  = [labels[i] for i in tr_idx]
    val_labels = [labels[i] for i in val_idx]

    tr_ds  = TweetDataset(tr_texts,  tr_labels)
    val_ds = TweetDataset(val_texts, val_labels)
    te_ds  = TweetDataset(test['clean_text'].tolist())

    tr_dl  = DataLoader(tr_ds,  batch_size=BATCH, shuffle=True,  num_workers=2)
    val_dl = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2)
    te_dl  = DataLoader(te_ds,  batch_size=BATCH, shuffle=False, num_workers=2)

    model     = SentimentRegressor().to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
    criterion = nn.MSELoss()

    best_rmse    = 999
    best_weights = None

    for epoch in range(EPOCHS):
        # Train
        model.train()
        for batch in tr_dl:
            ids  = batch['input_ids'].to(DEVICE)
            mask = batch['attention_mask'].to(DEVICE)
            tgt  = batch['target'].to(DEVICE)
            optimizer.zero_grad()
            pred = model(ids, mask)
            loss = criterion(pred, tgt)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        scheduler.step()

        # Validate
        model.eval()
        val_p = []
        with torch.no_grad():
            for batch in val_dl:
                out = model(batch['input_ids'].to(DEVICE),
                            batch['attention_mask'].to(DEVICE))
                val_p.extend(out.cpu().numpy())

        val_rmse = np.sqrt(np.mean((np.array(val_p) - np.array(val_labels))**2))
        print(f"  Epoch {epoch+1} | RMSE: {val_rmse:.4f}")

        if val_rmse < best_rmse:
            best_rmse    = val_rmse
            best_weights = model.state_dict()

    # OOF + Test predictions using best epoch weights
    model.load_state_dict(best_weights)
    model.eval()

    with torch.no_grad():
        vp = []
        for batch in val_dl:
            out = model(batch['input_ids'].to(DEVICE),
                        batch['attention_mask'].to(DEVICE))
            vp.extend(out.cpu().numpy())
        oof_preds[val_idx] = np.clip(vp, -1, 1)

        tp = []
        for batch in te_dl:
            out = model(batch['input_ids'].to(DEVICE),
                        batch['attention_mask'].to(DEVICE))
            tp.extend(out.cpu().numpy())
        test_preds += np.clip(tp, -1, 1) / N_FOLDS

overall_oof_rmse = np.sqrt(np.mean((oof_preds - np.array(labels))**2))
print(f"\n✅ Overall OOF RMSE: {overall_oof_rmse:.4f}")


========== FOLD 1/5 ==========


pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
classifier.dense.weight         | UNEXPECTED |  | 
classifier.out_proj.bias        | UNEXPECTED |  | 
classifier.out_proj.weight      | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
classifier.dense.bias           | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [1]:
submission = pd.DataFrame({
    'ID':     test['tweet_id'],
    'target': np.clip(test_preds, -1, 1)
})

submission.to_csv('submission1.csv', index=False)
print(submission.head(10))
print("\nSubmission shape:", submission.shape)
print("Target range:", submission['target'].min(), "to", submission['target'].max())

NameError: name 'pd' is not defined